# Load Waymax scenarios one by one

`scripts` 폴더의 기존 로딩 패턴(`viz/render.py`)을 따라, TFRecord에서 시나리오를 인덱스 기준으로 하나씩 읽어옵니다.

In [ ]:
from __future__ import annotations

import dataclasses
import sys
from pathlib import Path
from typing import Iterable

import numpy as np
from waymax import datatypes
from waymax.datatypes.roadgraph import MapElementIds

# Ensure repository root is importable when running this notebook from scripts/.
REPO_ROOT = Path.cwd().resolve().parents[0]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from waymax import config as waymax_config
from viz.render import _load_scenario_state_fast

import jax
from jax import numpy as jnp


def load_scenarios_one_by_one(
    tfrecord_path: str,
    scenario_indices: Iterable[int],
    *,
    max_num_objects: int = 32,
):
    """Yield (scenario_index, simulator_state) for each requested scenario."""
    ds_cfg = dataclasses.replace(
        waymax_config.WOD_1_3_1_TRAINING,
        path=str(tfrecord_path),
        max_num_objects=int(max_num_objects),
        batch_dims=(1,),
        shuffle_seed=0,
    )

    for scenario_index in scenario_indices:
        idx = int(scenario_index)
        if idx < 0:
            raise ValueError(f"scenario_index must be >= 0, got {idx}.")

        state = _load_scenario_state_fast(ds_cfg, idx)
        yield idx, state


def filter_lane_types(
    roadgraph_points: datatypes.RoadGraphPoints,
    lane_types_to_include: set[int],
):
    """Filter roadgraph points to include only specified lane types."""
    valid = roadgraph_points.valid
    mask = jnp.isin(roadgraph_points.types, jnp.asarray(list(lane_types_to_include))) & valid

    x = roadgraph_points.x[mask]
    y = roadgraph_points.y[mask]
    z = roadgraph_points.z[mask]
    dir_x = roadgraph_points.dir_x[mask]
    dir_y = roadgraph_points.dir_y[mask]
    dir_z = roadgraph_points.dir_z[mask]
    points_xyz = jnp.stack([x, y, z], axis=-1)
    points_dir_xyz = jnp.stack([dir_x, dir_y, dir_z], axis=-1)
    return points_xyz, points_dir_xyz


def build_incoming_outgoing_dicts(
    points_xyz: jnp.ndarray,
    points_dir_xyz: jnp.ndarray,
    *,
    distance_threshold_m: float = 1.0,
    min_forward_projection_m: float = 0.0,
) -> tuple[dict[int, list[int]], dict[int, list[int]]]:
    pts = np.asarray(points_xyz, dtype=np.float32)
    dirs = np.asarray(points_dir_xyz, dtype=np.float32)

    n = int(pts.shape[0])
    outgoing: dict[int, list[int]] = {i: [] for i in range(n)}
    incoming: dict[int, list[int]] = {i: [] for i in range(n)}

    xy = pts[:, :2]
    dir_xy = dirs[:, :2]

    for i in range(n):
        v = dir_xy[i]
        v_norm = float(np.linalg.norm(v))
        if v_norm < 1e-8:
            continue

        v_unit = v / v_norm
        diffs = xy - xy[i]  # [N, 2], vector from i to all points
        heading_diffs = (dir_xy * dir_xy[i][None, :]).sum(axis=-1) / (np.linalg.norm(dir_xy, axis=-1) * v_norm + 1e-8)
        heading_diffs = np.arccos(np.clip(heading_diffs, -1, 1))

        dists = np.linalg.norm(pts - pts[i][None, :], axis=1)
        proj = diffs @ v_unit  # signed projection on heading axis
        lateral = np.abs(diffs[:, 0] * v_unit[1] - diffs[:, 1] * v_unit[0])

        is_self = np.arange(n) == i
        candidate_mask = (
            (~is_self)
            & (dists <= float(distance_threshold_m))
        )
        outgoing_cand_mask = candidate_mask & (proj > float(min_forward_projection_m))
        incoming_cand_mask = candidate_mask & (proj < -float(min_forward_projection_m))

        outgoing_candidate_idx = np.where(outgoing_cand_mask)[0]
        incoming_candidate_idx = np.where(incoming_cand_mask)[0]

        # outgoing_cand_lateral = lateral[outgoing_candidate_idx]
        # incoming_cand_lateral = lateral[incoming_candidate_idx]
        outgoing_cand_lateral = heading_diffs[outgoing_candidate_idx]
        incoming_cand_lateral = heading_diffs[incoming_candidate_idx]

        outgoing_best_idx = outgoing_candidate_idx[np.argmin(outgoing_cand_lateral)] if outgoing_cand_lateral.size > 0 else None
        incoming_best_idx = incoming_candidate_idx[np.argmin(incoming_cand_lateral)] if incoming_cand_lateral.size > 0 else None

        if outgoing_best_idx is not None:
            outgoing[i].append(int(outgoing_best_idx))
            incoming[outgoing_best_idx].append(int(i))
        
        if incoming_best_idx is not None:
            outgoing[incoming_best_idx].append(int(i))
            incoming[i].append(int(incoming_best_idx))

    return incoming, outgoing


def adjacency_dict_to_csr(
    adjacency: dict[int, list[int]],
    *,
    num_nodes: int | None = None,
    sort_and_unique: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Convert adjacency dict {i: [j1, j2, ...]} into CSR arrays.

    Returns:
        offsets: int32 [N+1]
        indices: int32 [E]
    """
    if num_nodes is None:
        num_nodes = (max(adjacency.keys()) + 1) if adjacency else 0

    offsets = np.zeros((num_nodes + 1,), dtype=np.int32)
    flat_indices: list[int] = []

    for i in range(num_nodes):
        neigh = list(adjacency.get(i, []))
        if sort_and_unique:
            neigh = sorted(set(int(x) for x in neigh))
        else:
            neigh = [int(x) for x in neigh]

        flat_indices.extend(neigh)
        offsets[i + 1] = len(flat_indices)

    indices = np.asarray(flat_indices, dtype=np.int32)
    return offsets, indices


def incoming_outgoing_dicts_to_csr(
    incoming: dict[int, list[int]],
    outgoing: dict[int, list[int]],
    *,
    num_nodes: int | None = None,
) -> dict[str, np.ndarray]:
    """Convert incoming/outgoing dicts to CSR arrays."""
    if num_nodes is None:
        max_in = max(incoming.keys()) if incoming else -1
        max_out = max(outgoing.keys()) if outgoing else -1
        num_nodes = max(max_in, max_out) + 1

    incoming_offsets, incoming_indices = adjacency_dict_to_csr(
        incoming,
        num_nodes=num_nodes,
        sort_and_unique=True,
    )
    outgoing_offsets, outgoing_indices = adjacency_dict_to_csr(
        outgoing,
        num_nodes=num_nodes,
        sort_and_unique=True,
    )

    return {
        "incoming_offsets": incoming_offsets,
        "incoming_indices": incoming_indices,
        "outgoing_offsets": outgoing_offsets,
        "outgoing_indices": outgoing_indices,
    }

import matplotlib.pyplot as plt
import numpy as np


def visualize_lane_connectivity(
    points_xyz,
    incoming,
    outgoing,
    lane_idx: int,
    *,
    neighborhood_radius_m: float = 1000.0,
    figsize=(8, 8),
):
    """Visualize one lane point with its incoming/outgoing neighbors."""
    pts = np.asarray(points_xyz)
    n = int(pts.shape[0])

    if lane_idx < 0 or lane_idx >= n:
        raise ValueError(f"lane_idx must be in [0, {n - 1}], got {lane_idx}.")

    center_xy = pts[lane_idx, :2]
    rel = pts[:, :2] - center_xy[None, :]
    dist = np.linalg.norm(rel, axis=1)
    local_mask = dist <= float(neighborhood_radius_m)

    incoming_neighbors = sorted(set(int(i) for i in incoming.get(int(lane_idx), [])))
    outgoing_neighbors = sorted(set(int(i) for i in outgoing.get(int(lane_idx), [])))

    fig, ax = plt.subplots(figsize=figsize)

    # Background: nearby lane points
    # ax.scatter(
    #     road
    # )

    ax.scatter(
        pts[local_mask, 0],
        pts[local_mask, 1],
        s=1,
        c="lightgray",
        alpha=0.6,
        label="nearby lane points",
        zorder=1,
    )

    # Current lane point
    ax.scatter(
        pts[lane_idx, 0],
        pts[lane_idx, 1],
        s=120,
        c="black",
        marker="*",
        label=f"lane {lane_idx}",
        zorder=5,
    )


    # Incoming neighbors -> lane_idx (green arrows)
    visited = set()
    while len(incoming_neighbors) > 0:
        j = incoming_neighbors.pop(0)
        if j in visited:
            continue
        visited.add(j)
        x0, y0 = pts[j, 0], pts[j, 1]
        # x1, y1 = pts[current_idx, 0], pts[current_idx, 1]
        # ax.plot([x0, x1], [y0, y1], c="tab:green", lw=2, alpha=0.9, zorder=3)
        ax.scatter(x0, y0, s=1, c="tab:green", zorder=4)
        incoming_neighbors.extend(incoming.get(j, []))


    # lane_idx -> outgoing neighbors (blue arrows)
    visited = set()
    while len(outgoing_neighbors) > 0:
        j = outgoing_neighbors.pop(0)
        if j in visited:
            continue
        visited.add(j)
        x1, y1 = pts[j, 0], pts[j, 1]
        ax.scatter(x1, y1, s=1, c="tab:blue", zorder=4)
        outgoing_neighbors.extend(outgoing.get(j, []))

    ax.set_title(
        f"Lane {lane_idx} connectivity | incoming={len(incoming_neighbors)} | outgoing={len(outgoing_neighbors)}"
    )
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    # ax.axis("equal")
    ax.grid(True, alpha=0.2)
    ax.legend(loc="best")
    plt.show()


# Pick one lane index that has at least one incoming/outgoing edge.
def choose_connected_lane(incoming, outgoing, fallback: int = 0) -> int:
    keys = sorted(set(incoming.keys()) | set(outgoing.keys()))
    for k in keys:
        if len(incoming.get(k, [])) > 0 or len(outgoing.get(k, [])) > 0:
            return int(k)
    return int(fallback)


In [ ]:

# Example usage
TFRECORD_DIR = "/data/datasets/waymo/waymo-open-dataset-v1.3.1/tf_example/training"
TFRECORD_FILES = sorted(Path(TFRECORD_DIR).glob("training_tfexample.tfrecord-*"))

centerline_types = {
    int(MapElementIds.LANE_FREEWAY.value),
    int(MapElementIds.LANE_SURFACE_STREET.value),
    int(MapElementIds.LANE_BIKE_LANE.value),
}

for file_idx, tfrecord_file in enumerate(TFRECORD_FILES):
    print(f"file_index={file_idx} | tfrecord_file={tfrecord_file}")
    ds_cfg = dataclasses.replace(
        waymax_config.WOD_1_3_1_TRAINING,
        path=str(tfrecord_file),
        max_num_objects=32,
        batch_dims=(1,),
        shuffle_seed=0,
    )

    for scenario_idx in range(int(1e6)):
        try:
            state = _load_scenario_state_fast(ds_cfg, scenario_idx)
        except Exception as e:
            print(f"Max scenario index reached {scenario_idx}:", e)
            break

        points_xyz, points_dir_xyz = filter_lane_types(state.roadgraph_points, centerline_types)
        incoming_dict, outgoing_dict = build_incoming_outgoing_dicts(
            points_xyz,
            points_dir_xyz,
            distance_threshold_m=1.0,
        )
        csr = incoming_outgoing_dicts_to_csr(incoming_dict, outgoing_dict, num_nodes=int(points_xyz.shape[0]))

        print(
            f"scenario={scenario_idx} | num_points={points_xyz.shape[0]} | "
            f"incoming_edges={csr['incoming_indices'].shape[0]} | "
            f"outgoing_edges={csr['outgoing_indices'].shape[0]}"
        )
        print(
            f"sample point 0 outgoing (CSR): "
            f"{csr['outgoing_indices'][csr['outgoing_offsets'][0]:csr['outgoing_offsets'][1]].tolist()}"
        )
        
        lane_to_plot = choose_connected_lane(incoming_dict, outgoing_dict, fallback=10)
        print(
            f"Visualizing lane {lane_to_plot} | "
            f"incoming={incoming_dict.get(lane_to_plot, [])} | "
            f"outgoing={outgoing_dict.get(lane_to_plot, [])}"
        )

        visualize_lane_connectivity(
            points_xyz,
            incoming_dict,
            outgoing_dict,
            lane_idx=lane_to_plot,
        )
    break


